In [1]:

import sys
!{sys.executable} -m pip install pandas numpy

In [2]:
import sys
!{sys.executable} -m pip install scikit-learn

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [9]:
data = pd.read_csv(r"C:\Users\User\Downloads\UNSW_NB15_testing-set.csv")

In [10]:
print(data.head())

   id       dur proto service state  spkts  dpkts  sbytes  dbytes  \
0   1  0.000011   udp       -   INT      2      0     496       0   
1   2  0.000008   udp       -   INT      2      0    1762       0   
2   3  0.000005   udp       -   INT      2      0    1068       0   
3   4  0.000006   udp       -   INT      2      0     900       0   
4   5  0.000010   udp       -   INT      2      0    2126       0   

          rate  ...  ct_dst_sport_ltm  ct_dst_src_ltm  is_ftp_login  \
0   90909.0902  ...                 1               2             0   
1  125000.0003  ...                 1               2             0   
2  200000.0051  ...                 1               3             0   
3  166666.6608  ...                 1               3             0   
4  100000.0025  ...                 1               3             0   

   ct_ftp_cmd  ct_flw_http_mthd  ct_src_ltm  ct_srv_dst  is_sm_ips_ports  \
0           0                 0           1           2                0   
1     

In [8]:
import pandas as pd
import numpy as np

In [11]:
data = data.dropna()

In [13]:
encoder = LabelEncoder()



In [14]:
for col in data.select_dtypes(include=['object']).columns:
    data[col] = encoder.fit_transform(data[col])

In [15]:
X = data.drop("label", axis=1)

In [16]:
y = data["label"]

In [17]:
scaler = StandardScaler()

In [18]:
X_scaled = scaler.fit_transform(X)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)

In [21]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [22]:
class Autoencoder(nn.Module):

    def __init__(self, input_dim):

        super(Autoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

In [26]:
input_dim = X_train.shape[1]

In [32]:
class Autoencoder(nn.Module):

    def __init__(self, input_dim):

        super(Autoencoder, self).__init__()
    

In [34]:
class Autoencoder(nn.Module):

    def __init__(self, input_dim):

        super(Autoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):

        encoded = self.encoder(x)
        decoded = self.decoder(encoded)

        return decoded

In [35]:
input_dim = X_train.shape[1]

model = Autoencoder(input_dim)

In [36]:
criterion = nn.MSELoss()

In [37]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [38]:
epochs = 20

In [39]:
for epoch in range(epochs):

    # Forward pass
    outputs = model(X_train_tensor)

    loss = criterion(outputs, X_train_tensor)

    # Backward pass
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/20], Loss: 1.2542
Epoch [2/20], Loss: 1.2530
Epoch [3/20], Loss: 1.2517
Epoch [4/20], Loss: 1.2505
Epoch [5/20], Loss: 1.2493
Epoch [6/20], Loss: 1.2482
Epoch [7/20], Loss: 1.2470
Epoch [8/20], Loss: 1.2459
Epoch [9/20], Loss: 1.2447
Epoch [10/20], Loss: 1.2435
Epoch [11/20], Loss: 1.2424
Epoch [12/20], Loss: 1.2412
Epoch [13/20], Loss: 1.2400
Epoch [14/20], Loss: 1.2388
Epoch [15/20], Loss: 1.2376
Epoch [16/20], Loss: 1.2363
Epoch [17/20], Loss: 1.2350
Epoch [18/20], Loss: 1.2337
Epoch [19/20], Loss: 1.2323
Epoch [20/20], Loss: 1.2308


In [40]:
with torch.no_grad():

    reconstructed = model(X_test_tensor)

In [41]:
mse = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1)

In [42]:
threshold = mse.mean() + mse.std()

In [43]:
threshold = mse.mean() + mse.std()

In [44]:
predictions = (mse > threshold).int().numpy()

In [45]:
accuracy = accuracy_score(y_test, predictions)

In [46]:
print("\n==============================")
print("MODEL PERFORMANCE")
print("==============================")

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report")

print(classification_report(y_test, predictions))


MODEL PERFORMANCE
Accuracy: 0.4477

Classification Report
              precision    recall  f1-score   support

           0       0.45      0.99      0.62      7418
           1       0.35      0.01      0.01      9049

    accuracy                           0.45     16467
   macro avg       0.40      0.50      0.31     16467
weighted avg       0.39      0.45      0.28     16467



In [47]:
torch.save(model.state_dict(), "iot_autoencoder.pth")

print("\nModel saved successfully!")


Model saved successfully!


In [48]:
pip install streamlit torch pandas numpy scikit-learn

  Using cached protobuf-7.35.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
INFO: pip is looking at multiple versions of streamlit to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\User\\AppData\\Local\\Temp\\pip-unpack-e6k6bos7\\streamlit-1.56.0-py3-none-any.whl'
Consider using the `--user` option or check the permissions.

